<img src="../../images/logo/compactlogo.svg" width="200" alt="QENS">

# Tutorial 2: Syndrome Extraction — A Deep Dive

Syndrome extraction is the act of **measuring stabilizers** to learn where errors occurred — without measuring the logical qubit itself.

This notebook covers:
- What stabilizers are and why they detect errors
- How `compute_syndrome` works under the hood
- Single-qubit vs. multi-qubit error syndromes
- CSS code structure: X-type and Z-type syndromes
- Measurement errors and noisy syndrome extraction
- Visualizing syndromes on the code lattice

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Remove for inline display

from qens import RepetitionCode, SurfaceCode, ColorCode
from qens import DepolarizingError, BitFlipError, PhaseFlipError, MeasurementError
from qens import ComposedNoiseModel
from qens.core.types import PauliOp
from qens.utils.pauli_algebra import pauli_string_multiply

## 1. Stabilizer Generators

A stabilizer code is defined by a set of **stabilizer generators** — Pauli operators that:
1. All commute with each other
2. All fix the codewords (eigenvalue +1)

An error **anticommutes** with a stabilizer → flips its measurement from +1 to −1 → `syndrome bit = 1`.

In [ ]:
rep = RepetitionCode(distance=5)
stabs = rep.stabilizer_generators()

print(f"Repetition code d=5: {rep.num_data_qubits} data qubits, {len(stabs)} stabilizers")
print()
print("Stabilizer generators:")
for i, s in enumerate(stabs):
    pauli_labels = ['I','X','Y','Z']
    pauli_str = ''.join(pauli_labels[v] for v in s.pauli_string)
    print(f"  S{i}: {pauli_str}  (type={s.stabilizer_type}, qubits={s.qubits})")

The repetition code has `ZZ` stabilizers on adjacent qubits. An X error anticommutes with Z, so it triggers adjacent stabilizers.

## 2. Single-Qubit Error Syndromes

Let's trace exactly which stabilizers a single-qubit error triggers.

In [ ]:
print("Single X error syndromes on RepetitionCode(d=5):")
print(f"{'Qubit':>6s}  {'Error':>12s}  {'Syndrome':>12s}  {'Triggered':>15s}")
print("-" * 55)

for q in range(rep.num_data_qubits):
    error = np.zeros(rep.num_data_qubits, dtype=np.uint8)
    error[q] = PauliOp.X
    syndrome = rep.compute_syndrome(error)
    triggered = list(np.nonzero(syndrome)[0])
    print(f"  q={q}    {str(error):>12s}  {str(syndrome):>12s}  {str(triggered):>15s}")

**Key insight:** An error on an interior qubit triggers **two** adjacent stabilizers. An error on a boundary qubit triggers only **one**. The decoder uses this to infer error locations.

## 3. Multi-Qubit Errors and Syndrome Merging

Multiple errors can produce syndromes that **cancel** (XOR). Two adjacent X errors leave no syndrome at the shared boundary stabilizer.

In [ ]:
# Single error on qubit 1
e1 = np.zeros(5, dtype=np.uint8); e1[1] = PauliOp.X
s1 = rep.compute_syndrome(e1)

# Single error on qubit 2
e2 = np.zeros(5, dtype=np.uint8); e2[2] = PauliOp.X
s2 = rep.compute_syndrome(e2)

# Both errors together
e_both, _ = pauli_string_multiply(e1, e2)
s_both = rep.compute_syndrome(e_both)

print(f"Error on qubit 1:    {e1}  -> syndrome {s1}")
print(f"Error on qubit 2:    {e2}  -> syndrome {s2}")
print(f"Both errors:         {e_both}  -> syndrome {s_both}")
print()
print("Syndromes XOR:       ", s1 ^ s2)
print("(This equals the combined syndrome — stabilizer measurements are XOR.)")

## 4. Surface Code: X-type and Z-type Syndromes

CSS codes like the surface code separate X and Z errors:
- **X stabilizers** detect Z errors (they anticommute with Z)
- **Z stabilizers** detect X errors (they anticommute with X)

The syndrome vector concatenates both types.

In [ ]:
surf = SurfaceCode(distance=3)
stabs = surf.stabilizer_generators()

x_stabs = [s for s in stabs if s.stabilizer_type == 'X']
z_stabs = [s for s in stabs if s.stabilizer_type == 'Z']

print(f"Surface code d=3: {surf.num_data_qubits} data qubits")
print(f"  X-type stabilizers: {len(x_stabs)} (detect Z errors)")
print(f"  Z-type stabilizers: {len(z_stabs)} (detect X errors)")
print(f"  Total syndrome bits: {surf.num_ancilla_qubits}")

In [ ]:
# X error → triggers Z stabilizers
x_err = np.zeros(surf.num_data_qubits, dtype=np.uint8)
x_err[4] = PauliOp.X  # Center qubit

# Z error → triggers X stabilizers
z_err = np.zeros(surf.num_data_qubits, dtype=np.uint8)
z_err[4] = PauliOp.Z  # Center qubit

s_x = surf.compute_syndrome(x_err)
s_z = surf.compute_syndrome(z_err)

print(f"X error on qubit 4:")
print(f"  Syndrome: {s_x}")
print(f"  Triggered: {list(np.nonzero(s_x)[0])} (Z-type stabilizers)")
print()
print(f"Z error on qubit 4:")
print(f"  Syndrome: {s_z}")
print(f"  Triggered: {list(np.nonzero(s_z)[0])} (X-type stabilizers)")

**CSS independence:** X errors produce a syndrome entirely in the Z-stabilizer half, and Z errors in the X-stabilizer half. This allows X and Z errors to be decoded **independently**.

## 5. Y Errors: Both Syndromes Fire

A Y error is a combined X and Z error, so it triggers **both** X-type and Z-type stabilizers.

In [ ]:
y_err = np.zeros(surf.num_data_qubits, dtype=np.uint8)
y_err[4] = PauliOp.Y

s_y = surf.compute_syndrome(y_err)
print(f"Y error on qubit 4:")
print(f"  Syndrome: {s_y}")
print(f"  Triggered bits: {list(np.nonzero(s_y)[0])}")
print()
print("Compare with X⊕Z syndrome:")
print(f"  s_x XOR s_z = {s_x ^ s_z}")
print(f"  s_y         = {s_y}")
print(f"  Equal? {np.all(s_y == s_x ^ s_z)}")

## 6. Logical Errors: Zero Syndrome, Corrupted State

A **logical error** is a Pauli string that commutes with all stabilizers (zero syndrome) but changes the encoded information. This is the failure mode QEC protects against.

In [ ]:
# The logical X of the repetition code is X on all qubits
logical_x = np.ones(rep.num_data_qubits, dtype=np.uint8)  # X X X X X
syndrome_logical = rep.compute_syndrome(logical_x)

print(f"Logical X error:       {logical_x}")
print(f"Syndrome:              {syndrome_logical}  <-- all zeros!")
print(f"Is logical error:      {rep.is_logical_error(logical_x)}")

In [ ]:
# A correctable error has non-zero syndrome
correctable = np.zeros(rep.num_data_qubits, dtype=np.uint8)
correctable[2] = PauliOp.X

print(f"Correctable error:     {correctable}")
print(f"Syndrome:              {rep.compute_syndrome(correctable)}")
print(f"Is logical error:      {rep.is_logical_error(correctable)}")

## 7. Noisy Syndrome Extraction

In real hardware, syndrome measurements are themselves noisy. `MeasurementError` models ancilla qubit errors that flip syndrome bits independently of data qubit errors.

In [ ]:
from qens import NoisySampler, LookupTableDecoder

rep3 = RepetitionCode(distance=3)

# Compose data noise + measurement noise
data_noise = BitFlipError(p=0.02)
meas_noise = MeasurementError(p=0.01)
combined = ComposedNoiseModel([data_noise, meas_noise])

decoder = LookupTableDecoder(rep3)
sampler = NoisySampler(seed=0)

# Run with combined noise
result = sampler.run(rep3, combined, decoder, shots=5_000)
print(f"Combined noise (data=2%, meas=1%):")
print(f"  Logical error rate: {result.logical_error_rate:.4f}")

# Compare with data noise only
result_data_only = sampler.run(rep3, data_noise, decoder, shots=5_000)
print(f"\nData noise only (data=2%):")
print(f"  Logical error rate: {result_data_only.logical_error_rate:.4f}")

## 8. Syndrome Visualization

QENS can visualize the decoding graph, where nodes are stabilizers and highlighted nodes indicate triggered syndrome bits (defects).

In [ ]:
from qens import draw_decoding_graph, MWPMDecoder

surf3 = SurfaceCode(distance=3)
decoder = MWPMDecoder(surf3)
decoder.precompute()

# Inject an error and compute syndrome
error = np.zeros(surf3.num_data_qubits, dtype=np.uint8)
error[1] = PauliOp.X
error[4] = PauliOp.X
syndrome = surf3.compute_syndrome(error)

# Decode and inspect the matching
result = decoder.decode(syndrome)
print(f"Syndrome:   {syndrome}")
print(f"Defects at: {list(np.nonzero(syndrome)[0])}")
print(f"Matching:   {result.metadata['matching']}")

# Visualize
fig = draw_decoding_graph(decoder, syndrome=syndrome, title="Decoding Graph with Active Syndrome")
fig.show()

## Summary: Syndrome Extraction Pipeline

```
Physical error (Pauli string)
         ↓
compute_syndrome(error)   ← checks anticommutation with each stabilizer
         ↓
syndrome (binary vector)  ← 1 = stabilizer triggered (defect)
         ↓
decoder.decode(syndrome)  ← infers most likely correction
         ↓
correction (Pauli string) ← applied to recover codeword
         ↓
residual = error * correction
         ↓
is_logical_error(residual) ← did we fail?
```

**Key facts:**
- Syndrome is XOR-linear: `syndrome(A⊗B) = syndrome(A) XOR syndrome(B)`
- CSS codes separate X and Z syndromes — each decoded independently
- Logical errors have zero syndrome — they are undetectable; a good decoder avoids proposing them
- Measurement errors require repeated-round syndrome extraction (not yet in QENS v0)

**Next:** [Tutorial 3 — Decoder Comparison](03_decoder_comparison.ipynb)